<a href="https://colab.research.google.com/github/jahirxtrap/TIC_DKN_CORE_Dataset/blob/master/TIC_DKN_Modelos_tradicionales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabajo de Integración Curricular

## Librerías y rutas

In [2]:
# Librerías
!pip install recommenders --quiet

In [ ]:
!pip install pywFM --quiet
!git clone https://github.com/srendle/libfm /home/libfm && \
cd /home/libfm && \
git reset --hard 91f8504a15120ef6815d6e10cc7dee42eebaab0f && \
make all

Cloning into '/home/libfm'...
remote: Enumerating objects: 233, done.
remote: Total 233 (delta 0), reused 0 (delta 0), pack-reused 233 (from 1)
Receiving objects: 100% (233/233), 129.46 KiB | 16.18 MiB/s, done.
Resolving deltas: 100% (112/112), done.
HEAD is now at 91f8504 moved math.h below isinf so that Windows make works
cd src/libfm; make all
make[1]: Entering directory '/home/libfm/src/libfm'
g++ -O3 -Wall -c libfm.cpp -o libfm.o
mkdir -p ../../bin/
g++ -O3 -Wall libfm.o -o ../../bin/libFM
g++ -O3 -Wall -c tools/transpose.cpp -o tools/transpose.o
mkdir -p ../../bin/
g++ -O3 tools/transpose.o -o ../../bin/transpose
g++ -O3 -Wall -c tools/convert.cpp -o tools/convert.o
mkdir -p ../../bin/
g++ -O3 tools/convert.o -o ../../bin/convert
make[1]: Leaving directory '/home/libfm/src/libfm'


### Rutas

In [3]:
import os
import pandas as pd
import numpy as np

# Variables de entorno
os.environ["LIBFM_PATH"] = "/home/libfm/bin/"

# Datos
input_file = '/content/drive/MyDrive/TIC/data.jsonl'
output_file = '/content/drive/MyDrive/TIC/data.csv'
interactions_path = '/content/drive/MyDrive/TIC/interactions.csv'

# Dataset
inter_df = pd.read_csv(interactions_path)

# Carpeta de salida
output_dir = "/content/drive/MyDrive/TIC/dkn_output"
os.makedirs(output_dir, exist_ok=True)

# Archivo para las predicciones y resultados
predict_path = os.path.join(output_dir, "predict.txt")
result_path = os.path.join(output_dir, "result.txt")

# Archivos
train_file = os.path.join(output_dir, "train.txt")
valid_file = os.path.join(output_dir, "valid.txt")
test_file = os.path.join(output_dir, "test.txt")

# Ruta de los modelos
lfm_model_path = "/content/drive/MyDrive/TIC/dkn_output/model/LibFM"
dfm_model_path = "/content/drive/MyDrive/TIC/dkn_output/model/DFM"
os.makedirs(lfm_model_path, exist_ok=True)
os.makedirs(dfm_model_path, exist_ok=True)

# Archivos ffm
train_ffm = os.path.join(dfm_model_path, "train_ffm.txt")
valid_ffm = os.path.join(dfm_model_path, "valid_ffm.txt")
test_ffm = os.path.join(dfm_model_path, "test_ffm.txt")

## Evaluación

### Modelos tradicionales

#### Modelos LibFM

In [68]:
# Modelo LibFM
import numpy as np
from pywFM import FM
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Cargar datos desde archivo txt
def load_libfm_data(path):
    data = []
    labels = []
    with open(path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 3:
                continue
            label, user, item = parts
            labels.append(int(label))
            data.append({'user': user, 'item': item})
    return data, np.array(labels)

# Cargar datos
X_train_raw, y_train = load_libfm_data(train_file)
X_test_raw, y_test = load_libfm_data(test_file)

# Vectorizar datos
vec = DictVectorizer()
X_train = vec.fit_transform(X_train_raw)
X_test = vec.transform(X_test_raw)

# Entrenar modelo LibFM
fm = FM(
    task='classification',
    num_iter=32,
    k2=100,
    learn_rate=0.001,
    init_stdev=0.01,
    learning_method='sgd'
)

libfm_model = fm.run(X_train, y_train, X_test, y_test)
libfm_preds = libfm_model.predictions
libfm_binary = [1 if p >= 0.5 else 0 for p in libfm_preds]

# Evaluación
print("Resultado de evaluación (LibFM)")
print("- Precision:", f"{accuracy_score(y_test, libfm_binary)*100:.2f}%")
print("- F1 Score:", f"{f1_score(y_test, libfm_binary)*100:.2f}%")
print("- AUC:", f"{roc_auc_score(y_test, libfm_preds):.2f}")

Resultado de evaluación (LibFM)
- Precision: 75.94%
- F1 Score: 86.19%
- AUC: 0.47


#### Modelo DeepFM

In [37]:
# Preparar datos para DeepFM
import os

# Función para cargar datos desde archivo
def load_data(filepath):
    data = []
    with open(filepath, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 3:
                continue
            label, user, item = parts
            label = int(label)
            data.append((label, user, item))
    return data

# Crear mapeos únicos de usuarios e items a índices empezando en 1
all_users = set()
all_items = set()

for dataset in [train_data, valid_data, test_data]:
    for _, user, item in dataset:
        all_users.add(user)
        all_items.add(item)

user2id = {user: idx+1 for idx, user in enumerate(sorted(all_users))}
item2id = {item: idx+1 for idx, item in enumerate(sorted(all_items))}

print(f"Usuarios únicos: {len(user2id)}")
print(f"Items únicos: {len(item2id)}")

# Función para escribir archivo en formato FFM para xDeepFM
def write_ffm(data, filepath, user2id, item2id):
    with open(filepath, "w") as f:
        for label, user, item in data:
            # Saltar si user o item no están en el mapeo
            if user not in user2id or item not in item2id:
                continue
            u_idx = user2id[user]
            i_idx = item2id[item]
            # Field ids empiezan en 1 según xDeepFM
            # Field 1: usuario, Field 2: item
            line = f"{label} 1:{u_idx}:1 2:{i_idx}:1\n"
            f.write(line)

# Generar archivos ffm
write_ffm(train_file, train_ffm, user2id, item2id)
write_ffm(valid_file, valid_ffm, user2id, item2id)
write_ffm(test_file, test_ffm, user2id, item2id)
max_feature_index = max(max(user2id.values()), max(item2id.values())) + 1

print("Archivos FFM generados correctamente.")

Usuarios únicos: 1965
Items únicos: 12799
Archivos FFM generados correctamente.


In [42]:
# Modelo DeepFM
from recommenders.models.deeprec.models.xDeepFM import XDeepFMModel
from recommenders.models.deeprec.io.iterator import FFMTextIterator
from recommenders.models.deeprec.deeprec_utils import prepare_hparams

hparams = prepare_hparams(
    yaml_file=None,
    MODEL_DIR=dfm_model_path,
    metrics=['acc', 'f1', 'auc'],
    # General
    learning_rate=0.001,
    epochs=3,
    batch_size=2,
    show_step=200,
    model_type= 'xDeepFM',
    data_format="ffm",
    # XDeepFM
    FEATURE_COUNT=max_feature_index,
    FIELD_COUNT=2,
    method="classification",
    dim=100,
    layer_sizes=[128],
    cross_layer_sizes=[128],
    activation=["relu"],
    loss="cross_entropy_loss",
    user_dropout=False,
    cross_activation='identity',
    use_Linear_part=False,
    use_FM_part=True,
    use_CIN_part=True,
    use_DNN_part=False,
)

model_dfm = XDeepFMModel(hparams, FFMTextIterator)

Add FM part.
Add CIN part.
step 200 , total_loss: 0.6952, data_loss: 0.6952
step 400 , total_loss: 0.5738, data_loss: 0.5738
step 600 , total_loss: 0.5187, data_loss: 0.5187
step 800 , total_loss: 0.4768, data_loss: 0.4768
step 1000 , total_loss: 0.7413, data_loss: 0.7413
step 1200 , total_loss: 0.7516, data_loss: 0.7516
step 1400 , total_loss: 0.7576, data_loss: 0.7576
step 1600 , total_loss: 0.7783, data_loss: 0.7783
step 1800 , total_loss: 0.3472, data_loss: 0.3472
step 2000 , total_loss: 0.3352, data_loss: 0.3352
step 2200 , total_loss: 0.3140, data_loss: 0.3140
step 2400 , total_loss: 0.3060, data_loss: 0.3060
step 2600 , total_loss: 0.2946, data_loss: 0.2946
step 2800 , total_loss: 0.2807, data_loss: 0.2807
step 3000 , total_loss: 0.2749, data_loss: 0.2749
step 3200 , total_loss: 0.2695, data_loss: 0.2695
step 3400 , total_loss: 0.8546, data_loss: 0.8546
step 3600 , total_loss: 0.2686, data_loss: 0.2686
step 3800 , total_loss: 0.8616, data_loss: 0.8616
step 4000 , total_loss: 0.2

In [ ]:
# Entrenar el modelo
model_dfm.fit(train_ffm, valid_ffm)

In [46]:
# Guardar el modelo
model_dfm.saver.save(model_dfm.sess, os.path.join(dfm_model_path, 'dfm_model.ckpt'))
print("Modelo guardado en:", dfm_model_path)

Modelo guardado en: /content/drive/MyDrive/TIC/dkn_output/model/DFM


In [48]:
# Cargar el modelo
model_dfm.load_model(os.path.join(dfm_model_path, 'dfm_model.ckpt'))
print("Modelo cargado")

Modelo cargado


In [69]:
# Evaluación
result_dfm = model_dfm.run_eval(test_ffm)
print("Resultado de evaluación (DeepFM)")
print(f"- Precision: {result_dfm['acc']:.2%}")
print(f"- F1 Score: {result_dfm['f1']:.2%}")
print(f"- AUC: {result_dfm['auc']:.2f}")

Resultado de evaluación (DeepFM)
- Precision: 79.28%
- F1 Score: 88.41%
- AUC: 0.49
